In [1]:
# --------------------------
# Required Libraries
# --------------------------
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")


In [ ]:

df= pd.read_csv((r"./datasets/bookings_train.csv"))

# --------------------------
# Ensure datetime types
# --------------------------
df['check_in_time'] = pd.to_datetime(df['check_in_time'])
df['check_out_time'] = pd.to_datetime(df['check_out_time'])
df['appointment_date'] = pd.to_datetime(df['appointment_date'])
df['appointment_time'] = pd.to_datetime(df['appointment_time'], format='%H:%M')

# Drop rows with missing critical times
df = df.dropna(subset=['check_in_time', 'check_out_time'])

# Calculate completion time in minutes
df['completion_time'] = (df['check_out_time'] - df['check_in_time']).dt.total_seconds() / 60

# --------------------------
# Task overall mean
# --------------------------
task_mean_df = df.groupby('task_id')['completion_time'].mean().reset_index()
task_mean_df.rename(columns={'completion_time':'task_mean_time'}, inplace=True)
df = df.merge(task_mean_df, on='task_id', how='left')

# --------------------------
# Previous same-day mean completion time
# --------------------------
df = df.sort_values(['task_id','appointment_date','appointment_time'])
df['prev_completion_time'] = df.groupby(['task_id','appointment_date'])['completion_time'] \
                                .transform(lambda x: x.shift().expanding().mean())

# Fill missing with task mean
df['prev_completion_time'] = df['prev_completion_time'].fillna(df['task_mean_time'])


# --------------------------
# Previous day same-hour mean (rolling_prev_day_mean)
# --------------------------
# Extract hour and day_of_week
df['hour'] = df['appointment_time'].dt.hour
df['minute'] = df['appointment_time'].dt.minute
df['day_of_week'] = df['appointment_date'].dt.dayofweek
df['month'] = df['appointment_date'].dt.month

# Prepare previous day lookup
prev_day_df = df[['task_id','appointment_date','hour','completion_time']].copy()
prev_day_df['appointment_date'] = prev_day_df['appointment_date'] + pd.Timedelta(days=1)
prev_day_df.rename(columns={'completion_time':'rolling_prev_day_mean'}, inplace=True)

# Merge previous day data
df = df.merge(prev_day_df[['task_id','appointment_date','hour','rolling_prev_day_mean']],
              on=['task_id','appointment_date','hour'], how='left')

# Fill missing with task + day_of_week mean
task_dow_mean = df.groupby(['task_id','day_of_week'])['completion_time'].mean().reset_index()
task_dow_mean.rename(columns={'completion_time':'task_dow_mean'}, inplace=True)
df = df.merge(task_dow_mean, on=['task_id','day_of_week'], how='left')
df['rolling_prev_day_mean'] = df['rolling_prev_day_mean'].fillna(df['task_dow_mean'])
df.drop(columns=['task_dow_mean'], inplace=True)

# --------------------------
# Features & Target
# --------------------------
categorical_features = ['task_id','hour','minute','day_of_week','month']
numeric_features = ['task_mean_time','prev_completion_time','rolling_prev_day_mean']

X = df[numeric_features + categorical_features]
y = df['completion_time']

# --------------------------
# Train-test split
# --------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)


In [3]:

# --------------------------
# Preprocessing
# --------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num', StandardScaler(), numeric_features)
    ]
)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)


In [4]:
from xgboost import XGBRegressor
# Train XGBoost
# --------------------------
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_proc, y_train)

# --------------------------
# Evaluate XGBoost
# --------------------------
y_pred_xgb = xgb_model.predict(X_test_proc)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("\nXGBoost Performance:")
print(f"MAE: {mae_xgb:.2f} minutes")
print(f"RMSE: {rmse_xgb:.2f} minutes")
print(f"R²: {r2_xgb:.3f}")


XGBoost Performance:
MAE: 9.59 minutes
RMSE: 13.97 minutes
R²: 0.678


In [5]:
import optuna
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBRegressor
import numpy as np

# --------------------------
# Define objective function for Optuna
# --------------------------
def objective(trial):
    # Hyperparameters to tune
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'random_state': 42,
        'n_jobs': -1
    }
    
    # Define model
    model = XGBRegressor(**params)
    
    # Use 5-fold cross-validation
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    # Use negative RMSE as scoring (since Optuna maximizes objective)
    score = -cross_val_score(model, X_train_proc, y_train, cv=kf, scoring='neg_root_mean_squared_error').mean()
    
    return score

# --------------------------
# Run Optuna study
# --------------------------
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20)  # You can increase n_trials for better tuning

print("Best hyperparameters:", study.best_params)
print("Best RMSE (CV):", study.best_value)

# --------------------------
# Train XGBoost with best hyperparameters
# --------------------------
best_params = study.best_params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train_proc, y_train)


[I 2025-08-23 13:55:23,445] A new study created in memory with name: no-name-1e6fdef9-adad-404d-a060-a2b3998bfee3
[I 2025-08-23 13:55:45,658] Trial 0 finished with value: 12.984920819642833 and parameters: {'n_estimators': 250, 'max_depth': 12, 'learning_rate': 0.1205712628744377, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'gamma': 0.7799726016810132, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676}. Best is trial 0 with value: 12.984920819642833.
[I 2025-08-23 13:56:20,495] Trial 1 finished with value: 14.409788427825427 and parameters: {'n_estimators': 341, 'max_depth': 10, 'learning_rate': 0.010725209743171996, 'subsample': 0.9849549260809971, 'colsample_bytree': 0.9162213204002109, 'gamma': 1.0616955533913808, 'reg_alpha': 0.9091248360355031, 'reg_lambda': 0.9170225492671691}. Best is trial 0 with value: 12.984920819642833.
[I 2025-08-23 13:56:34,002] Trial 2 finished with value: 14.43538685717368 and parameters: {'n_estimators': 222, 

Best hyperparameters: {'n_estimators': 410, 'max_depth': 12, 'learning_rate': 0.1347033543076489, 'subsample': 0.812134809681067, 'colsample_bytree': 0.5349462362555917, 'gamma': 1.7555887348012431, 'reg_alpha': 2.40980044304833, 'reg_lambda': 3.435837126284375}
Best RMSE (CV): 12.492180403362037


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.5349462362555917, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=1.7555887348012431, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1347033543076489,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=12, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=410, n_jobs=None,
             num_parallel_tree=None, ...)

In [6]:
# --------------------------
# Evaluate on training set
# --------------------------
y_train_pred = best_model.predict(X_train_proc)
mae_train = mean_absolute_error(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
r2_train = r2_score(y_train, y_train_pred)

print("Tuned XGBoost Performance on TRAINING set:")
print(f"MAE: {mae_train:.2f} minutes")
print(f"RMSE: {rmse_train:.2f} minutes")
print(f"R²: {r2_train:.3f}")

# --------------------------
# Evaluate on test set
# --------------------------
y_test_pred = best_model.predict(X_test_proc)
mae_test = mean_absolute_error(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_test = r2_score(y_test, y_test_pred)

print("\nTuned XGBoost Performance on TEST set:")
print(f"MAE: {mae_test:.2f} minutes")
print(f"RMSE: {rmse_test:.2f} minutes")
print(f"R²: {r2_test:.3f}")




Tuned XGBoost Performance on TRAINING set:
MAE: 6.00 minutes
RMSE: 8.34 minutes
R²: 0.885

Tuned XGBoost Performance on TEST set:
MAE: 7.97 minutes
RMSE: 11.82 minutes
R²: 0.769
